# 24 · Evaluation, Interpretability, Robustness, Efficiency and Responsible AI

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anuxlab/deep-learning-systems/blob/main/notebooks/24-evaluation-robustness-efficiency/24_evaluation_robustness_efficiency.ipynb)
[![Run on GitHub Actions](https://img.shields.io/badge/run-GitHub%20Actions-2088FF?logo=githubactions&logoColor=white)](https://github.com/anuxlab/deep-learning-systems/actions/workflows/nb-24-evaluation-robustness-efficiency.yml)

| | |
|---|---|
| **Wiki chapter** | [Module 24](https://github.com/anuxlab/deep-learning-systems/wiki/Module-24) |
| **Level** | Advanced |
| **Typical CPU runtime** | ~1–2 min (GitHub free runner, no GPU) |
| **Prerequisites** | Modules 07, 10, 22 |

## What you will do

- Compute classification metrics from scratch (confusion matrix, F1, MCC, ROC-AUC, PR-AUC) and see why accuracy misleads on imbalanced data.
- Report uncertainty: bootstrap confidence intervals and paired model comparison.
- Measure and fix calibration (reliability diagram, ECE, temperature scaling).
- Explain predictions: permutation importance, saliency and Integrated Gradients (with the completeness check).
- Attack and defend: FGSM, adversarial training, corruption robustness.
- Compress models: int8 quantisation, magnitude pruning, bf16; and audit fairness metrics; write a model card.

> **How to use this notebook.** Read each markdown cell, *predict* what the next code cell will print, then run it.
> Every section ends with `check(...)` assertions: if one fails, something in your understanding (or the code) is off.
> Figures are saved to `figures/` and a machine-readable summary to `metrics.json` next to this notebook.

## 0 · Setup and reproducibility

We import shared helpers (`show` saves each figure to `figures/`, `check` asserts and logs, `timer` logs wall-clock time), fix all random seeds and print an environment report so every run is traceable (see the wiki's *Implementation and Research Standards*).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "_common"))   # shared helpers: notebooks/_common/dlsys.py
from dlsys import *                                       # np, plt, torch, seed_everything, show, check, timer, ...

SEED = seed_everything(42)
ENV = env_report()

## 1 · Metrics from scratch — and the accuracy paradox
For a binary classifier: $\mathrm{precision}=\frac{TP}{TP+FP}$, $\mathrm{recall}=\frac{TP}{TP+FN}$, $F_1=\frac{2PR}{P+R}$, MCC$=\frac{TP\,TN-FP\,FN}{\sqrt{(TP+FP)(TP+FN)(TN+FP)(TN+FN)}}$. **ROC-AUC** = probability that a random positive is scored above a random negative (rank statistic); **PR-AUC** (average precision) focuses on the positive class and is far more informative under heavy imbalance. Dataset: synthetic, **1% positives**.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn import metrics as skm
Xi, yi = make_classification(n_samples=20000, n_features=10, n_informative=4, weights=[.99], flip_y=.01, class_sep=1.0, random_state=0)
Xtr, Xte, ytr, yte = train_test_split(Xi, yi, test_size=.5, random_state=0, stratify=yi); clf = LogisticRegression(max_iter=2000).fit(Xtr, ytr); s = clf.predict_proba(Xte)[:, 1]; pred = (s > .5).astype(int)
print(f"positives in test set: {yte.mean():.3%}")
TP = int(((pred == 1) & (yte == 1)).sum()); FP = int(((pred == 1) & (yte == 0)).sum()); FN = int(((pred == 0) & (yte == 1)).sum()); TN = int(((pred == 0) & (yte == 0)).sum())
prec, rec = TP / max(TP + FP, 1), TP / (TP + FN); f1 = 2 * prec * rec / (prec + rec + 1e-12); mcc = (TP * TN - FP * FN) / np.sqrt(float(TP + FP) * (TP + FN) * (TN + FP) * (TN + FN))
def auroc(y, s): r = skm.rankdata(s) if hasattr(skm, "rankdata") else __import__("scipy.stats").stats.rankdata(s); n1, n0 = y.sum(), (1 - y).sum(); return (r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)
def avg_prec(y, s): o = np.argsort(-s); yy = y[o]; tp = np.cumsum(yy); p_ = tp / np.arange(1, len(yy) + 1); return (p_ * yy).sum() / yy.sum()
print(f"confusion  TP={TP} FP={FP} FN={FN} TN={TN}"); print(f"accuracy {(TP+TN)/len(yte):.4f} | precision {prec:.3f} recall {rec:.3f} F1 {f1:.3f} MCC {mcc:.3f} | ROC-AUC {auroc(yte, s):.3f} PR-AUC {avg_prec(yte, s):.3f}")
check(np.isclose(prec, skm.precision_score(yte, pred)) and np.isclose(rec, skm.recall_score(yte, pred)) and np.isclose(f1, skm.f1_score(yte, pred)), "precision / recall / F1 match sklearn")
check(np.isclose(mcc, skm.matthews_corrcoef(yte, pred)), "MCC matches sklearn"); check(np.isclose(auroc(yte, s), skm.roc_auc_score(yte, s)), "rank-based ROC-AUC matches sklearn"); check(np.isclose(avg_prec(yte, s), skm.average_precision_score(yte, s)), "average precision matches sklearn")
dummy_acc = 1 - yte.mean(); print(f"\naccuracy paradox: predicting 'negative' for everyone gets accuracy {dummy_acc:.4f}, F1 {skm.f1_score(yte, np.zeros_like(yte)):.2f}, recall 0 !")
check(dummy_acc > .98 and skm.f1_score(yte, np.zeros_like(yte)) == 0, "a useless constant classifier has >98% accuracy: never report accuracy alone on imbalanced data")
fig, ax = plt.subplots(1, 2, figsize=(11, 4)); fpr, tpr, _ = skm.roc_curve(yte, s); pr, rc, _ = skm.precision_recall_curve(yte, s)
ax[0].plot(fpr, tpr); ax[0].plot([0, 1], [0, 1], "k:"); ax[0].set_xlabel("FPR"); ax[0].set_ylabel("TPR"); ax[0].set_title(f"ROC (AUC {skm.roc_auc_score(yte, s):.2f})"); ax[1].plot(rc, pr); ax[1].axhline(yte.mean(), color="k", ls=":", label="chance = prevalence"); ax[1].set_xlabel("recall"); ax[1].set_ylabel("precision"); ax[1].legend(); ax[1].set_title(f"PR (AP {skm.average_precision_score(yte, s):.2f})"); show("roc_pr_curves")
METRICS["imbalanced"] = dict(f1=f1, mcc=mcc, roc_auc=auroc(yte, s), pr_auc=avg_prec(yte, s), constant_classifier_accuracy=dummy_acc)

## 2 · Baselines, bootstrap confidence intervals, paired comparison
A number without uncertainty is not a result. The **bootstrap** resamples the test set with replacement to approximate the sampling distribution of a metric. To compare two models, resample the *same* indices for both (a **paired** bootstrap) and look at the distribution of the *difference*. Baselines: majority class and a trivial single-feature rule.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(100, random_state=0, n_jobs=1, min_samples_leaf=3).fit(Xtr, ytr); s_rf = rf.predict_proba(Xte)[:, 1]; s_dummy = DummyClassifier(strategy="prior").fit(Xtr, ytr).predict_proba(Xte)[:, 1]
rng = np.random.default_rng(0); B = 500; idx_b = [rng.integers(0, len(yte), len(yte)) for _ in range(B)]
def boot(fn, *ss): return np.array([fn(yte[i], *[s_[i] for s_ in ss]) for i in idx_b])
ap = lambda y, s_: skm.average_precision_score(y, s_)
for nm, s_ in [("majority baseline", s_dummy), ("logistic regression", s), ("random forest", s_rf)]:
    v = boot(ap, s_); print(f"{nm:20s} PR-AUC {ap(yte, s_):.3f}  95% CI [{np.percentile(v, 2.5):.3f}, {np.percentile(v, 97.5):.3f}]")
diff = boot(ap, s_rf) - boot(ap, s); lo, hi = np.percentile(diff, [2.5, 97.5]); print(f"paired difference (RF − LR) in PR-AUC: {diff.mean():+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]  → {'significant' if lo > 0 or hi < 0 else 'NOT significant (CI contains 0)'}")
check(ap(yte, s) > 3 * ap(yte, s_dummy), "model beats the majority baseline by a wide margin"); check(np.percentile(boot(ap, s), 97.5) - np.percentile(boot(ap, s), 2.5) > .02, "with only ~100 positives the CI is wide — report it")
plt.hist(diff, 40); plt.axvline(0, color="k"); plt.xlabel("PR-AUC(RF) − PR-AUC(LR)"); plt.title("Paired bootstrap of the difference"); show("paired_bootstrap"); METRICS["paired_diff_ci"] = [float(lo), float(hi)]

## 3 · Calibration
A model is **calibrated** if among predictions with confidence $p$, a fraction $p$ are correct. **ECE** $=\sum_b\frac{|B_b|}{n}\,|\mathrm{acc}(B_b)-\mathrm{conf}(B_b)|$ (bin-weighted gap). Modern networks trained to low training loss are typically **over-confident**. **Temperature scaling** divides logits by a scalar $T>1$ fitted on a validation set by minimising NLL — it changes confidence but *not* the arg-max, so accuracy is unchanged. We deliberately over-train an MLP on a small digits subset.

In [ ]:
from sklearn.datasets import load_digits
D = load_digits(); Xd = torch.tensor(D.data / 16., dtype=torch.float32); yd = torch.tensor(D.target); ix = np.random.default_rng(0).permutation(len(Xd)); trn, val, tst = ix[:300], ix[300:800], ix[800:]
torch.manual_seed(0); mlp = nn.Sequential(nn.Linear(64, 256), nn.ReLU(), nn.Linear(256, 256), nn.ReLU(), nn.Linear(256, 10)); o = torch.optim.Adam(mlp.parameters(), 3e-3)
for ep in range(300): perm = torch.randperm(len(trn)); [(o.zero_grad(), F.cross_entropy(mlp(Xd[trn][perm[s:s+32]]), yd[trn][perm[s:s+32]]).backward(), o.step()) for s in range(0, len(trn), 32)]
mlp.eval()
def ece(probs, labels, bins=10):
    conf, pred = probs.max(1); correct = (pred == labels).float(); e = 0.; rows = []
    for lo in np.linspace(0, 1, bins + 1)[:-1]:
        m = (conf > lo) & (conf <= lo + 1 / bins)
        if m.any(): e += m.float().mean() * abs(correct[m].mean() - conf[m].mean()); rows.append((conf[m].mean().item(), correct[m].mean().item(), m.sum().item()))
    return float(e), rows
with torch.no_grad(): lv, lt = mlp(Xd[val]), mlp(Xd[tst])
T_ = torch.ones(1, requires_grad=True); opt = torch.optim.LBFGS([T_], lr=.1, max_iter=200)
def closure(): opt.zero_grad(); l = F.cross_entropy(lv / T_, yd[val]); l.backward(); return l
opt.step(closure); T = T_.item(); print(f"fitted temperature T = {T:.3f}")
res = {}
for nm, lg in [("raw", lt), ("temperature-scaled", lt / T)]:
    p_ = F.softmax(lg, 1); e, rows = ece(p_, yd[tst]); res[nm] = dict(ece=e, nll=F.cross_entropy(lg, yd[tst]).item(), acc=(lg.argmax(1) == yd[tst]).float().mean().item(), rows=rows, conf=p_.max(1)[0].mean().item()); print(f"{nm:20s} acc {res[nm]['acc']:.3f}  mean confidence {res[nm]['conf']:.3f}  ECE {e:.4f}  NLL {res[nm]['nll']:.4f}")
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
for a_, nm in zip(ax, res): r_ = np.array(res[nm]["rows"]); a_.plot([0, 1], [0, 1], "k:"); a_.bar(r_[:, 0], r_[:, 1], width=.08, alpha=.6, edgecolor="k"); a_.set_xlabel("confidence"); a_.set_ylabel("accuracy"); a_.set_title(f"{nm}: ECE {res[nm]['ece']:.3f}")
show("reliability_diagram")
check(T > 1, "T > 1: the over-trained network was over-confident"); check(res["temperature-scaled"]["nll"] < res["raw"]["nll"], "temperature scaling lowers test NLL"); check(res["temperature-scaled"]["ece"] < res["raw"]["ece"], "…and lowers ECE")
check(abs(res["temperature-scaled"]["acc"] - res["raw"]["acc"]) < 1e-9, "…without changing accuracy (arg-max is invariant to a global scale)"); METRICS["calibration"] = {k: {a: b for a, b in v.items() if a != "rows"} for k, v in res.items()}

## 4 · Interpretability
**Permutation importance:** shuffle one feature (a pixel) at test time; the accuracy drop measures reliance on it. **Saliency:** $|\partial f_c/\partial x|$. **Integrated Gradients** (Sundararajan et al. 2017): $\mathrm{IG}_i=(x_i-x'_i)\int_0^1\partial_if(x'+\alpha(x-x'))\,d\alpha$, approximated by a Riemann sum. Its **completeness axiom**: $\sum_i\mathrm{IG}_i=f(x)-f(x')$ — an exact, checkable property. We also run the *sanity check of Adebayo et al.*: saliency from a model with **randomised weights** should look different from a trained model's.

In [ ]:
x = Xd[tst][:1]; cls = int(yd[tst][0]); f = lambda z: F.softmax(mlp(z), 1)[:, cls]; base = torch.zeros_like(x)
def integrated_gradients(x, base, steps=200):
    al = torch.linspace(0, 1, steps)[:, None]; pts = (base + al * (x - base)).requires_grad_(); f(pts).sum().backward(); return (x - base) * pts.grad.mean(0, keepdim=True)
ig = integrated_gradients(x, base); gap = (f(x) - f(base)).item(); print(f"sum of IG attributions {ig.sum().item():.5f}   f(x)-f(baseline) {gap:.5f}")
check(abs(ig.sum().item() - gap) < 5e-3, "completeness: IG attributions sum to f(x) − f(baseline)")
xs = x.clone().requires_grad_(); f(xs).sum().backward(); sal = xs.grad.abs()
torch.manual_seed(99); rnd = nn.Sequential(nn.Linear(64, 256), nn.ReLU(), nn.Linear(256, 256), nn.ReLU(), nn.Linear(256, 10)); xr = x.clone().requires_grad_(); F.softmax(rnd(xr), 1)[:, cls].sum().backward(); sal_rnd = xr.grad.abs()
corr = np.corrcoef(sal.flatten().numpy(), sal_rnd.flatten().numpy())[0, 1]; print(f"correlation between trained-model saliency and random-model saliency: {corr:.2f}")
check(corr < .6, "sanity check: saliency depends on the learned weights (not just the input)")
# permutation importance on pixels
base_acc = (mlp(Xd[tst]).argmax(1) == yd[tst]).float().mean().item(); imp = np.zeros(64); g = torch.Generator().manual_seed(0)
for j in range(64):
    d_ = []
    for _ in range(5): Xp = Xd[tst].clone(); Xp[:, j] = Xp[torch.randperm(len(Xp), generator=g), j]; d_.append(base_acc - (mlp(Xp).argmax(1) == yd[tst]).float().mean().item())
    imp[j] = np.mean(d_)
fig, ax = plt.subplots(1, 4, figsize=(14, 3.4)); ax[0].imshow(x.view(8, 8), cmap="gray"); ax[0].set_title(f"input (class {cls})"); ax[1].imshow(sal.view(8, 8), cmap="hot"); ax[1].set_title("saliency |∂f/∂x|"); ax[2].imshow(ig.view(8, 8), cmap="RdBu", vmin=-abs(ig).max(), vmax=abs(ig).max()); ax[2].set_title("Integrated Gradients"); ax[3].imshow(imp.reshape(8, 8), cmap="viridis"); ax[3].set_title("permutation importance (acc drop)")
for a_ in ax: a_.axis("off")
show("attributions"); check(imp.max() > 0 and imp[[0, 7, 56, 63]].mean() < imp.mean() + 1e-9, "corner pixels (almost always blank in digits) are unimportant")

## 5 · Adversarial robustness and corruptions
**FGSM:** $x_{adv}=x+\epsilon\,\mathrm{sign}(\nabla_x\mathcal L)$ — a single step in the direction that most increases the loss under an $\ell_\infty$ budget $\epsilon$. **Adversarial training** minimises the loss on such perturbed inputs (a min–max problem). We compare a standard and an adversarially trained MLP on digits, and also test *natural* corruptions (Gaussian noise).

In [ ]:
trn2 = ix[:1200]; tst2 = ix[1200:]
def train_clf(adv_eps=0., epochs=60, seed=0):
    torch.manual_seed(seed); m = nn.Sequential(nn.Linear(64, 256), nn.ReLU(), nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, 10)); o = torch.optim.Adam(m.parameters(), 2e-3)
    for ep in range(epochs):
        perm = torch.as_tensor(trn2)[torch.randperm(len(trn2))]
        for s_ in range(0, len(perm), 64):
            b = perm[s_:s_ + 64]; xb = Xd[b]
            if adv_eps: xb = fgsm(m, xb, yd[b], adv_eps)
            o.zero_grad(); F.cross_entropy(m(xb), yd[b]).backward(); o.step()
    return m.eval()
def fgsm(m, x, y, eps): x = x.clone().requires_grad_(); F.cross_entropy(m(x), y).backward(); return (x + eps * x.grad.sign()).clamp(0, 1).detach()
def acc_at(m, eps):
    X_, y_ = Xd[tst2], yd[tst2]; Xa = fgsm(m, X_, y_, eps) if eps else X_
    with torch.no_grad(): return (m(Xa).argmax(1) == y_).float().mean().item()
with timer("adv_training"): std, adv = train_clf(0.), train_clf(.15)
epsx = [0, .05, .1, .15, .2, .3]; A = {n: [acc_at(m, e) for e in epsx] for n, m in [("standard", std), ("adversarially trained (ε=0.15)", adv)]}
for n, v in A.items(): print(f"{n:32s}", " ".join(f"ε={e}:{a:.2f}" for e, a in zip(epsx, v)))
for n, v in A.items(): plt.plot(epsx, v, "o-", label=n)
plt.xlabel("FGSM ε (ℓ∞, pixel range [0,1])"); plt.ylabel("accuracy"); plt.legend(); plt.title("Adversarial robustness"); show("fgsm_robustness")
check(A["standard"][0] > .9, "standard model is accurate on clean data"); check(A["standard"][3] < .5 * A["standard"][0], "a small ε=0.15 perturbation destroys the standard model's accuracy")
check(A["adversarially trained (ε=0.15)"][3] > A["standard"][3] + .2, "adversarial training substantially improves robustness at the training ε")
print(f"clean accuracy: standard {A['standard'][0]:.3f} vs adversarially trained {A['adversarially trained (ε=0.15)'][0]:.3f}.  NOTE: the robustness–accuracy trade-off is often reported on large natural-image datasets; on tiny, easy digits the perturbations act as a regulariser and no clean-accuracy cost is visible. Do not generalise this to harder tasks.")
check(A["adversarially trained (ε=0.15)"][0] > A["standard"][0] - .05, "no large clean-accuracy penalty on this easy dataset")
g = torch.Generator().manual_seed(0); noise = {s_: [(std(((Xd[tst2] + s_ * torch.randn(Xd[tst2].shape, generator=g)).clamp(0, 1))).argmax(1) == yd[tst2]).float().mean().item(), (adv(((Xd[tst2] + s_ * torch.randn(Xd[tst2].shape, generator=g)).clamp(0, 1))).argmax(1) == yd[tst2]).float().mean().item()] for s_ in [0, .1, .2, .3, .5]}
print("Gaussian-noise accuracy [standard, adv-trained]:", {k: np.round(v, 2).tolist() for k, v in noise.items()}); METRICS["fgsm"] = {k: v for k, v in A.items()}

## 6 · Efficiency: quantisation, pruning, reduced precision
- **Post-training int8 quantisation** (symmetric, per-tensor): $q=\mathrm{round}(w/s)$, $s=\max|w|/127$; store int8 + one float scale (≈4× smaller). We implement it by hand (so you see exactly what happens) and compare with `torch.ao.quantization.quantize_dynamic` if available.
- **Magnitude pruning:** zero the smallest-|w| fraction globally; accuracy is flat up to moderate sparsity (one-shot, no fine-tuning) and then collapses; prune–fine-tune cycles reach far higher sparsity.
- **bf16:** 8 exponent bits (same range as fp32) but 7 mantissa bits ⇒ ~2–3 significant digits.

In [ ]:
import copy, time, io
big = train_clf(0., epochs=60); acc0 = acc_at(big, 0); nparam = count_params(big)
def int8_quant(m):
    q = copy.deepcopy(m); sizes = 0
    for l in q:
        if isinstance(l, nn.Linear): s_ = l.weight.abs().max() / 127; w8 = torch.round(l.weight / s_).clamp(-127, 127).to(torch.int8); l.weight.data = w8.float() * s_; sizes += w8.numel() + 4
    return q, sizes
q, bytes_int8 = int8_quant(big); acc_q = acc_at(q, 0); fp32_bytes = sum(p.numel() for n_, p in big.named_parameters() if "weight" in n_) * 4
print(f"fp32 weights {fp32_bytes/1024:.1f} KiB -> int8 {bytes_int8/1024:.1f} KiB (×{fp32_bytes/bytes_int8:.1f} smaller) | accuracy {acc0:.3f} -> {acc_q:.3f}")
check(fp32_bytes / bytes_int8 > 3.9, "int8 weights are ~4× smaller"); check(abs(acc_q - acc0) < .02, "int8 quantisation costs < 2 points of accuracy")
try:
    import warnings; warnings.filterwarnings("ignore"); dq = torch.ao.quantization.quantize_dynamic(copy.deepcopy(big), {nn.Linear}, dtype=torch.qint8); print("torch dynamic int8 quantisation accuracy:", round(acc_at(dq, 0), 3))
except Exception as e: print("torch.ao dynamic quantisation unavailable here:", type(e).__name__)
def prune_global(m, sparsity):
    p = copy.deepcopy(m); ws = torch.cat([l.weight.abs().flatten() for l in p if isinstance(l, nn.Linear)]); thr = torch.quantile(ws, sparsity)
    for l in p:
        if isinstance(l, nn.Linear): l.weight.data[l.weight.abs() < thr] = 0
    return p
sps = [0, .5, .8, .9, .95, .98, .99]; pa = [acc_at(prune_global(big, s_), 0) for s_ in sps]
plt.plot([s_ * 100 for s_ in sps], pa, "o-"); plt.xlabel("global sparsity (%)"); plt.ylabel("accuracy"); plt.title("Magnitude pruning (no fine-tuning)"); show("pruning")
print(dict(zip(sps, np.round(pa, 3)))); print("(one-shot pruning WITHOUT fine-tuning: accuracy holds to ~50% sparsity then degrades; iterative prune-and-fine-tune reaches much higher sparsity — see exercise.)")
check(pa[1] > acc0 - .01, "50% of the weights can be removed with < 1 point of accuracy loss"); check(pa[-1] < pa[1] - .3, "extreme sparsity destroys the model unless it is fine-tuned")
b16 = big[0].weight.data.to(torch.bfloat16).float(); rel = ((b16 - big[0].weight.data).abs() / big[0].weight.data.abs().clamp(min=1e-8)).median().item(); print(f"bf16 median relative rounding error of weights: {rel:.4f} (~2^-8 = {2**-8:.4f})")
check(rel < 2 ** -7, "bf16 keeps ≈ 3 significant digits"); big16 = copy.deepcopy(big).to(torch.bfloat16)
with torch.no_grad(): a16 = (big16(Xd[tst2].to(torch.bfloat16)).argmax(1) == yd[tst2]).float().mean().item()
print(f"bf16 inference accuracy {a16:.3f} vs fp32 {acc0:.3f}"); check(abs(a16 - acc0) < .02, "bf16 inference matches fp32 accuracy")
METRICS["efficiency"] = dict(params=nparam, acc_fp32=acc0, acc_int8=acc_q, acc_bf16=a16, pruning=dict(zip(map(str, sps), pa)))

## 7 · Fairness audit
Two groups ($a$, $b$) with different base rates and a feature distribution shifted for $b$. Common group metrics for a classifier $\hat Y$: **demographic parity** (equal positive-prediction rates), **equal opportunity** (equal TPR), **equalised odds** (equal TPR *and* FPR). These generally **cannot all hold** when base rates differ (impossibility results of Kleinberg et al. and Chouldechova). We measure the gaps, then apply group-specific thresholds (post-processing) and show the trade-off.

In [ ]:
rng = np.random.default_rng(1); n = 20000; grp = rng.random(n) < .4                                                     # True = group b (minority)
base = np.where(grp, .25, .45); yg = (rng.random(n) < base).astype(int); xg = rng.normal(size=(n, 3)) + yg[:, None] * np.array([1.2, .8, .5]) - grp[:, None] * np.array([.5, 0, 0])      # feature shift for group b
tr_, te_ = np.arange(n) < n // 2, np.arange(n) >= n // 2; clf = LogisticRegression().fit(xg[tr_], yg[tr_]); sg = clf.predict_proba(xg[te_])[:, 1]; yt, gt = yg[te_], grp[te_]
def rates(thr_a, thr_b): pr = np.where(gt, sg > thr_b, sg > thr_a); out = {}; 
def group_report(thr_a, thr_b):
    pr = np.where(gt, sg > thr_b, sg > thr_a); r = {}
    for nm, m in [("a", ~gt), ("b", gt)]: r[nm] = dict(pos_rate=pr[m].mean(), tpr=pr[m & (yt == 1)].mean(), fpr=pr[m & (yt == 0)].mean(), acc=(pr[m] == yt[m]).mean())
    return r, (pr == yt).mean()
r0, acc_all = group_report(.5, .5); print(f"single threshold 0.5 | overall acc {acc_all:.3f}")
for g_ in "ab": print(f"  group {g_}: positive-rate {r0[g_]['pos_rate']:.3f}  TPR {r0[g_]['tpr']:.3f}  FPR {r0[g_]['fpr']:.3f}  acc {r0[g_]['acc']:.3f}")
dp0 = abs(r0["a"]["pos_rate"] - r0["b"]["pos_rate"]); eo0 = abs(r0["a"]["tpr"] - r0["b"]["tpr"]); print(f"demographic-parity gap {dp0:.3f}   equal-opportunity gap {eo0:.3f}")
check(dp0 > .05, "demographic parity is violated (different base rates and feature shift)")
best = min(((abs(group_report(.5, tb)[0]["a"]["tpr"] - group_report(.5, tb)[0]["b"]["tpr"]), tb) for tb in np.linspace(.15, .6, 46))); tb = best[1]; r1, acc1 = group_report(.5, tb)
print(f"group-b threshold {tb:.2f} equalises opportunity: TPR gap {abs(r1['a']['tpr']-r1['b']['tpr']):.3f} (was {eo0:.3f}); overall accuracy {acc_all:.3f} -> {acc1:.3f}; FPR gap {abs(r1['a']['fpr']-r1['b']['fpr']):.3f}; DP gap {abs(r1['a']['pos_rate']-r1['b']['pos_rate']):.3f}")
check(abs(r1["a"]["tpr"] - r1["b"]["tpr"]) < eo0, "group-specific threshold reduces the equal-opportunity gap")
check(abs(r1["a"]["fpr"] - r1["b"]["fpr"]) > abs(r0["a"]["fpr"] - r0["b"]["fpr"]) - .005 or acc1 <= acc_all + 1e-3, "fairness criteria trade off against each other / accuracy (no free lunch)")
thr_grid = np.linspace(.15, .6, 46); gaps = [abs(group_report(.5, t)[0]["a"]["tpr"] - group_report(.5, t)[0]["b"]["tpr"]) for t in thr_grid]; accs_ = [group_report(.5, t)[1] for t in thr_grid]
fig, ax = plt.subplots(figsize=(6.5, 4)); ax.plot(thr_grid, gaps, label="TPR gap"); ax.plot(thr_grid, [abs(group_report(.5, t)[0]["a"]["fpr"] - group_report(.5, t)[0]["b"]["fpr"]) for t in thr_grid], label="FPR gap"); ax.plot(thr_grid, [abs(group_report(.5, t)[0]["a"]["pos_rate"] - group_report(.5, t)[0]["b"]["pos_rate"]) for t in thr_grid], label="positive-rate gap"); ax.set_xlabel("group-b threshold"); ax.set_ylabel("gap"); ax.legend(); ax2 = ax.twinx(); ax2.plot(thr_grid, accs_, "k--"); ax2.set_ylabel("overall accuracy (dashed)"); ax.set_title("Fairness criteria vs threshold"); show("fairness_tradeoff")
METRICS["fairness"] = dict(dp_gap=dp0, eo_gap=eo0, eo_gap_after=abs(r1["a"]["tpr"] - r1["b"]["tpr"]), acc_before=acc_all, acc_after=acc1)

## 8 · Reproducibility artefacts: run manifest and model card
The wiki's research standards ask for a record of *what was run and what it achieved*. We assemble a machine-readable manifest (already in `metrics.json`) and a human-readable **model card** (`model_card.md`, Mitchell et al. 2019) for the digits classifier evaluated above, including its limitations.

In [ ]:
card = f'''# Model card — digits MLP (notebook 24)

**Model:** 64-256-128-10 MLP, {nparam:,} parameters, trained with Adam for 60 epochs on 1,200 8×8 digit images (scikit-learn `load_digits`).
**Intended use:** teaching/demo of evaluation practice. **Not** for any real decision-making.

## Metrics (held-out 597 images)
| metric | value |
|---|---|
| clean accuracy (fp32) | {acc0:.3f} |
| int8-quantised accuracy | {acc_q:.3f} |
| accuracy under FGSM ε=0.15 (standard) | {A['standard'][3]:.3f} |
| accuracy under FGSM ε=0.15 (adversarially trained variant) | {A['adversarially trained (ε=0.15)'][3]:.3f} |
| expected calibration error (raw → temperature-scaled, other model) | {res['raw']['ece']:.3f} → {res['temperature-scaled']['ece']:.3f} |

## Limitations & caveats
- Tiny, clean, low-resolution dataset; results do not transfer to natural images or distribution shift.
- Brittle to small adversarial perturbations unless adversarially trained (which costs clean accuracy).
- Single seed; confidence intervals not reported for these figures (see section 2 for how to compute them).
- Fairness metrics were illustrated on a *synthetic* two-group problem and say nothing about this model.
'''
open("model_card.md", "w").write(card); print(card)
import platform; METRICS["manifest"] = dict(python=platform.python_version(), torch=torch.__version__, seed=SEED, notebook="24")
check(len(card) > 500, "model card written to model_card.md")

In [ ]:
save_metrics("24", METRICS)
log.info('notebook 24 finished OK')

## Wrap-up

## Exercises (do these next)

1. Add class-conditional (per-digit) metrics and a confusion-matrix heatmap; which digits are confused most?
2. Implement PGD (multi-step FGSM with projection) and show that FGSM-trained models are weaker against it.
3. Add post-training static quantisation to 4 bits and find the accuracy cliff; implement iterative magnitude pruning with fine-tuning and compare the sparsity cliff.

## Where to read more

- Wiki chapter: [Module 24](https://github.com/anuxlab/deep-learning-systems/wiki/Module-24) (derivations, worked example, Q&A).
- Guo et al. (2017) *On calibration of modern neural networks*; Sundararajan et al. (2017) IG; Adebayo et al. (2018) *Sanity checks for saliency maps*; Goodfellow et al. (2015) FGSM; Madry et al. (2018) PGD; Han et al. (2016) *Deep compression*; Mitchell et al. (2019) *Model cards*; Kleinberg et al. (2017).